# MealSense DINOv3 Dish Classifier

Train and evaluate a DINOv3 image classifier for dish recognition. The notebook is organized to run in Google Colab or locally on Windows.


## Workflow

1. Install the Python dependencies.
2. Configure the dataset, local DINOv3 model, and output directories.
3. Discover dish classes and create train, validation, and test splits.
4. Train a frozen-backbone classifier head.
5. Optionally fine-tune the final DINOv3 blocks.
6. Evaluate the model and export the checkpoint and configuration.

### Data layout

The dish dataset should contain one folder per class inside `food_dataset/train`, for example:

```text
food_dataset/
└── train/
    ├── bakorkhani/
    ├── beguni/
    └── ...
```

In Colab, set `MEALSENSE_DISH_ROOT` or edit the configuration cell to point to the mounted Drive folder. On Windows, the notebook uses the workspace `food_dataset` and `dinov3_offline_model` folders by default. Set `MEALSENSE_DISH_ROOT` or `MEALSENSE_DINOV3_PATH` when using a different location.


In [9]:
# Run this cell once per environment.
%pip install -q numpy torch torchvision transformers accelerate scikit-learn pillow tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Environment and paths

The cell mounts Google Drive only when running in Colab. Local Windows runs use ordinary filesystem paths and environment variables; no Colab-only imports are required.


In [10]:
import os
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT = Path.cwd()
DEFAULT_DISH_ROOT = (
    Path("/content/drive/MyDrive/Mealsense/food_dataset")
    if IS_COLAB
    else PROJECT_ROOT / "food_dataset"
)
DEFAULT_DINOV3_PATH = (
    Path("/content/drive/MyDrive/Mealsense/dinov3_offline_model")
    if IS_COLAB
    else PROJECT_ROOT / "dinov3_offline_model"
)
DEFAULT_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "meal_classifier_dinov3"

DISH_ROOT = Path(os.environ.get("MEALSENSE_DISH_ROOT",
                 str(DEFAULT_DISH_ROOT))).expanduser()
DINOV3_PATH = Path(os.environ.get("MEALSENSE_DINOV3_PATH",
                   str(DEFAULT_DINOV3_PATH))).expanduser()
OUTPUT_DIR = Path(os.environ.get("MEALSENSE_OUTPUT_DIR",
                  str(DEFAULT_OUTPUT_DIR))).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runtime: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dish dataset: {DISH_ROOT}")
print(f"DINOv3 model: {DINOV3_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

Runtime: Local
Dish dataset: d:\CS\Hackathon\Mealsense_Models\food_dataset
DINOv3 model: d:\CS\Hackathon\Mealsense_Models\dinov3_offline_model
Output directory: d:\CS\Hackathon\Mealsense_Models\outputs\meal_classifier_dinov3


## Imports and reproducibility


In [11]:
import json
import random
import shutil
import time

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORKERS = 0 if os.name == "nt" else 2
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

Device: cpu


## Dataset discovery and splits


In [12]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
TRAIN_DIR = DISH_ROOT / "train"


def class_folders(root):
    return sorted(path.name for path in root.iterdir() if path.is_dir())


def samples_from_folder(root, class_to_index):
    samples = []
    for class_name, class_index in class_to_index.items():
        class_dir = root / class_name
        if class_dir.exists():
            samples.extend(
                (str(image_path), class_index)
                for image_path in class_dir.rglob("*")
                if image_path.is_file() and image_path.suffix.lower() in IMG_EXTS
            )
    return samples


if not TRAIN_DIR.is_dir():
    raise FileNotFoundError(
        f"Missing training directory: {TRAIN_DIR}. "
        "Set MEALSENSE_DISH_ROOT to the folder containing the train directory."
    )

DISH_CLASSES = class_folders(TRAIN_DIR)
if len(DISH_CLASSES) < 2:
    raise ValueError(
        f"Expected at least two dish class folders; found {DISH_CLASSES}")

DISH_TO_IDX = {name: index for index, name in enumerate(DISH_CLASSES)}
IDX_TO_DISH = {index: name for name, index in DISH_TO_IDX.items()}
ALL_SAMPLES = samples_from_folder(TRAIN_DIR, DISH_TO_IDX)
if not ALL_SAMPLES:
    raise ValueError(f"No supported images found in {TRAIN_DIR}")

train_val_samples, test_samples = train_test_split(
    ALL_SAMPLES,
    test_size=0.15,
    random_state=SEED,
    stratify=[label for _, label in ALL_SAMPLES],
)
train_samples, val_samples = train_test_split(
    train_val_samples,
    test_size=0.15 / 0.85,
    random_state=SEED,
    stratify=[label for _, label in train_val_samples],
)

print(f"Classes: {len(DISH_CLASSES)}")
print(
    f"Train/validation/test: {len(train_samples)}/{len(val_samples)}/{len(test_samples)}")

Classes: 9
Train/validation/test: 1254/269/269


## Transforms and data loaders


In [13]:
IMAGE_SIZE = 224
BATCH_SIZE = 16

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(0.12, 0.12, 0.10, 0.03),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class DishDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        with Image.open(image_path) as image:
            image_tensor = self.transform(image.convert("RGB"))
        return image_tensor, label


train_loader = DataLoader(
    DishDataset(train_samples, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=WORKERS,
    pin_memory=DEVICE == "cuda",
)
val_loader = DataLoader(
    DishDataset(val_samples, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=DEVICE == "cuda",
)
test_loader = DataLoader(
    DishDataset(test_samples, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=DEVICE == "cuda",
)

for name, index in DISH_TO_IDX.items():
    print(
        f"{name}: train={sum(label == index for _, label in train_samples)}, "
        f"val={sum(label == index for _, label in val_samples)}, "
        f"test={sum(label == index for _, label in test_samples)}"
    )

bakorkhani: train=108, val=23, test=23
beguni: train=180, val=38, test=39
biriyani: train=107, val=23, test=23
chickpeas: train=147, val=32, test=32
egg_omlete: train=143, val=30, test=31
fuchka: train=110, val=24, test=23
haleem: train=111, val=24, test=24
hilsha_fish: train=180, val=39, test=38
kacchi: train=168, val=36, test=36


## Load the DINOv3 backbone


In [14]:
if not DINOV3_PATH.exists():
    raise FileNotFoundError(
        f"DINOv3 model directory not found: {DINOV3_PATH}. "
        "Set MEALSENSE_DINOV3_PATH to a local model directory."
    )

backbone = AutoModel.from_pretrained(
    str(DINOV3_PATH),
    local_files_only=True,
)
print(f"DINOv3 hidden size: {backbone.config.hidden_size}")

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 3501.16it/s]

DINOv3 hidden size: 384


## Build the classifier


In [15]:
class DINOClassifier(nn.Module):
    def __init__(self, backbone, number_of_classes):
        super().__init__()
        self.backbone = backbone
        hidden_size = backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(0.25),
            nn.Linear(hidden_size, number_of_classes),
        )

    def forward(self, pixel_values):
        features = self.backbone(
            pixel_values=pixel_values).last_hidden_state[:, 0]
        return self.head(features)


dish_model = DINOClassifier(backbone, len(DISH_CLASSES)).to(DEVICE)
for parameter in dish_model.backbone.parameters():
    parameter.requires_grad = False

## Train the classifier head

The backbone is frozen in this stage. The best validation macro-F1 checkpoint is saved to the configured output directory.


In [16]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)


def run_epoch(loader, optimizer=None):
    is_training = optimizer is not None
    dish_model.train(is_training)
    total_loss = 0.0
    labels = []
    predictions = []

    for images, targets in tqdm(loader, leave=False):
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)
        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = dish_model(images)
            loss = criterion(logits, targets)
            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * len(targets)
        labels.extend(targets.detach().cpu().tolist())
        predictions.extend(logits.argmax(dim=1).detach().cpu().tolist())

    return (
        total_loss / len(loader.dataset),
        accuracy_score(labels, predictions),
        f1_score(labels, predictions, average="macro"),
        labels,
        predictions,
    )


EPOCHS = 10
optimizer = torch.optim.AdamW(
    dish_model.head.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)
CHECKPOINT_PATH = OUTPUT_DIR / "dinov3_dish_best.pth"
best_val_f1 = -1.0

for epoch in range(EPOCHS):
    start_time = time.time()
    train_metrics = run_epoch(train_loader, optimizer)
    val_metrics = run_epoch(val_loader)
    print(
        f"Epoch {epoch + 1}/{EPOCHS}: "
        f"train_acc={train_metrics[1]:.3f} | "
        f"val_acc={val_metrics[1]:.3f} | "
        f"val_macro_f1={val_metrics[2]:.3f} | "
        f"{time.time() - start_time:.1f}s"
    )

    if val_metrics[2] > best_val_f1:
        best_val_f1 = val_metrics[2]
        torch.save(
            {
                "state_dict": dish_model.state_dict(),
                "dish_to_idx": DISH_TO_IDX,
                "idx_to_dish": IDX_TO_DISH,
                "model_path": str(DINOV3_PATH),
                "image_size": IMAGE_SIZE,
                "val_macro_f1": best_val_f1,
            },
            CHECKPOINT_PATH,
        )

print(f"Best checkpoint: {CHECKPOINT_PATH}")

Epoch 1/10: train_acc=0.624 | val_acc=0.799 | val_macro_f1=0.794 | 72.0s


Epoch 2/10: train_acc=0.826 | val_acc=0.822 | val_macro_f1=0.818 | 67.8s


Epoch 3/10: train_acc=0.867 | val_acc=0.829 | val_macro_f1=0.823 | 73.3s


Epoch 4/10: train_acc=0.881 | val_acc=0.844 | val_macro_f1=0.837 | 67.1s


Epoch 5/10: train_acc=0.887 | val_acc=0.855 | val_macro_f1=0.851 | 73.8s


Epoch 6/10: train_acc=0.895 | val_acc=0.874 | val_macro_f1=0.870 | 67.1s


Epoch 7/10: train_acc=0.910 | val_acc=0.870 | val_macro_f1=0.867 | 72.3s


Epoch 8/10: train_acc=0.917 | val_acc=0.874 | val_macro_f1=0.870 | 74.9s


Epoch 9/10: train_acc=0.915 | val_acc=0.866 | val_macro_f1=0.861 | 68.7s


Epoch 10/10: train_acc=0.923 | val_acc=0.877 | val_macro_f1=0.872 | 68.1s
Best checkpoint: d:\CS\Hackathon\Mealsense_Models\outputs\meal_classifier_dinov3\dinov3_dish_best.pth


## Optional DINOv3 fine-tuning

Set `RUN_FINE_TUNING` to `True` to unfreeze the final two transformer blocks. This is more expensive and is disabled by default.


In [17]:
RUN_FINE_TUNING = False
FINE_TUNING_EPOCHS = 3

if RUN_FINE_TUNING:
    for parameter in dish_model.backbone.parameters():
        parameter.requires_grad = False

    unlocked_parameters = 0
    for name, parameter in dish_model.backbone.named_parameters():
        if ".10." in name or ".11." in name:
            parameter.requires_grad = True
            unlocked_parameters += 1

    fine_tune_optimizer = torch.optim.AdamW(
        [
            {
                "params": [
                    parameter
                    for parameter in dish_model.backbone.parameters()
                    if parameter.requires_grad
                ],
                "lr": 1e-5,
            },
            {"params": dish_model.head.parameters(), "lr": 1e-4},
        ],
        weight_decay=1e-4,
    )

    print(f"Unlocked parameter tensors: {unlocked_parameters}")
    for epoch in range(FINE_TUNING_EPOCHS):
        train_metrics = run_epoch(train_loader, fine_tune_optimizer)
        val_metrics = run_epoch(val_loader)
        print(
            f"Fine-tuning epoch {epoch + 1}/{FINE_TUNING_EPOCHS}: "
            f"train_acc={train_metrics[1]:.3f} | "
            f"val_acc={val_metrics[1]:.3f} | "
            f"val_macro_f1={val_metrics[2]:.3f}"
        )

        if val_metrics[2] > best_val_f1:
            best_val_f1 = val_metrics[2]
            torch.save(
                {
                    "state_dict": dish_model.state_dict(),
                    "dish_to_idx": DISH_TO_IDX,
                    "idx_to_dish": IDX_TO_DISH,
                    "model_path": str(DINOV3_PATH),
                    "image_size": IMAGE_SIZE,
                    "val_macro_f1": best_val_f1,
                },
                CHECKPOINT_PATH,
            )
else:
    print("Fine-tuning skipped.")

Fine-tuning skipped.


## Evaluate on the test split


In [18]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
dish_model.load_state_dict(checkpoint["state_dict"])
test_metrics = run_epoch(test_loader)
print(f"Test loss: {test_metrics[0]:.4f}")
print(f"Test accuracy: {test_metrics[1]:.4f}")
print(f"Test macro-F1: {test_metrics[2]:.4f}")
print(
    classification_report(
        test_metrics[3],
        test_metrics[4],
        target_names=[IDX_TO_DISH[index]
                      for index in range(len(DISH_CLASSES))],
        zero_division=0,
    )
)

Test loss: 0.6962
Test accuracy: 0.8476
Test macro-F1: 0.8346
              precision    recall  f1-score   support

  bakorkhani       0.84      0.91      0.88        23
      beguni       0.92      0.85      0.88        39
    biriyani       0.81      0.91      0.86        23
   chickpeas       0.78      0.88      0.82        32
  egg_omlete       0.86      0.77      0.81        31
      fuchka       0.83      0.87      0.85        23
      haleem       0.67      0.50      0.57        24
 hilsha_fish       0.88      0.97      0.93        38
      kacchi       0.94      0.89      0.91        36

    accuracy                           0.85       269
   macro avg       0.84      0.84      0.83       269
weighted avg       0.85      0.85      0.84       269



## Export the trained model

The export contains the best classifier checkpoint and a JSON configuration. The output path works on both Windows and Colab.


In [19]:
export_dir = OUTPUT_DIR / "export"
export_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(CHECKPOINT_PATH, export_dir / CHECKPOINT_PATH.name)

config = {
    "classifier": "DINOv3",
    "dinov3_model_path": str(DINOV3_PATH),
    "image_size": IMAGE_SIZE,
    "dish_to_idx": DISH_TO_IDX,
    "idx_to_dish": IDX_TO_DISH,
    "best_validation_macro_f1": best_val_f1,
}
with (export_dir / "config.json").open("w", encoding="utf-8") as config_file:
    json.dump(config, config_file, indent=2)

archive_path = shutil.make_archive(
    str(OUTPUT_DIR / "meal_classifier_dinov3"), "zip", export_dir)
print(f"Export directory: {export_dir}")
print(f"Archive: {archive_path}")

Export directory: d:\CS\Hackathon\Mealsense_Models\outputs\meal_classifier_dinov3\export
Archive: d:\CS\Hackathon\Mealsense_Models\outputs\meal_classifier_dinov3\meal_classifier_dinov3.zip
